In [1]:
from src.data import (
    EntityDatabase,
    WikidataClient,
    WikipediaClient,
    WESTERN_COUNTRIES,
    ARABIC_COUNTRIES,
    ASIAN_COUNTRIES,
)
from src.utils import setup_logging
import logging
import json

# Setup logging
setup_logging(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize clients
db = EntityDatabase("data/entities.db")
wikidata = WikidataClient(rate_limit_delay=1.0)
wikipedia = WikipediaClient(language='en', rate_limit_delay=0.1)


def has_complete_labels(entity, entity_type_key='person'):
    """
    Check if entity has complete labels in en, fr, ar
    Arabic can be either 'ar' or 'arz' (fallback)
    """
    required_keys = [
        f'{entity_type_key}Label_en',
        f'{entity_type_key}Label_fr',
    ]

    # Check en and fr
    for key in required_keys:
        if not entity.get(key):
            return False

    # Check ar or arz for Arabic
    ar_key = f'{entity_type_key}Label_ar'
    if not entity.get(ar_key):
        return False  # Arabic fallback already applied by wikidata_client

    return True


def collect_people(nationalities, limit=50, min_year=1850, max_year=2000):
    """Collect people entities with full metadata"""
    logger.info(f"Collecting {limit} people...")

    # Query Wikidata
    people = wikidata.get_people_multilingual(
        nationalities=nationalities,
        limit=limit,
        min_year=min_year,
        max_year=max_year
    )

    logger.info(f"Got {len(people)} people from Wikidata")

    # Filter for complete labels
    valid_people = [p for p in people if has_complete_labels(p, 'person')]
    logger.info(f"Filtered to {len(valid_people)} with complete labels")

    collected = 0
    for person in valid_people:
        wikidata_id = person.get('person_id')
        if not wikidata_id:
            continue

        # Determine culture from nationality
        nationality_id = person.get('nationality_id')
        if nationality_id in WESTERN_COUNTRIES:
            culture = 'Western'
        elif nationality_id in ARABIC_COUNTRIES:
            culture = 'Arabic'
        elif nationality_id in ASIAN_COUNTRIES:
            culture = 'Asian'
        else:
            culture = 'Other'

        # Add base entity
        entity_id = db.add_entity(
            wikidata_id=wikidata_id,
            entity_type='person',
            culture=culture,
            country=person.get('nationalityLabel_en')
        )

        if not entity_id:
            continue

        # Add labels in all languages
        for lang in ['en', 'fr', 'ar']:
            label = person.get(f'personLabel_{lang}')
            if label:
                db.add_entity_label(entity_id, lang, label)

        # Add metadata
        if person.get('gender'):
            db.add_metadata(entity_id, 'gender', person['gender'])

        if person.get('occupation'):
            # Store all occupations as JSON list
            db.add_metadata(entity_id, 'occupation', json.dumps([person['occupation']]))

        if person.get('birthDate'):
            birth_year = person['birthDate'][:4]  # Extract year
            db.add_metadata(entity_id, 'birth_year', birth_year)

        if person.get('deathDate'):
            death_year = person['deathDate'][:4]
            db.add_metadata(entity_id, 'death_year', death_year)
            db.add_metadata(entity_id, 'is_alive', 'False')
        else:
            db.add_metadata(entity_id, 'is_alive', 'True')

        # Fetch Wikipedia language count separately (faster than SPARQL)
        num_wiki_langs = wikidata.get_sitelinks_count(wikidata_id)
        db.add_metadata(entity_id, 'num_wikipedia_languages', str(num_wiki_langs))

        # Get exact Wikipedia titles from Wikidata (more reliable than searching!)
        wikipedia_titles = wikidata.get_wikipedia_titles(wikidata_id)

        # Fetch Wikipedia using exact titles
        wiki_pages = {}
        for lang in ['en', 'fr', 'ar']:
            if lang in wikipedia_titles:
                wikipedia.change_language(lang)
                page_data = wikipedia.get_page_by_title(wikipedia_titles[lang])

                if page_data:
                    # Add backlinks count to metadata
                    backlinks = wikipedia.get_backlinks(page_data['title'])
                    page_data['metadata']['num_backlinks'] = backlinks
                    wiki_pages[lang] = page_data
                else:
                    wiki_pages[lang] = None
            else:
                logger.info(f"No {lang} Wikipedia sitelink for {wikidata_id}")
                wiki_pages[lang] = None

        # Add facts (birth date, birth place, nationality)
        # Birth date
        if person.get('birthDate'):
            fact_id = db.add_fact(entity_id, 'P569', person['birthDate'])

        # Birth place (URL + multilingual labels)
        if person.get('birthPlace'):
            fact_id = db.add_fact(entity_id, 'P19', person['birthPlace'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'birthPlaceLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Nationality
        if person.get('nationality'):
            fact_id = db.add_fact(entity_id, 'P27', person['nationality'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = person.get(f'nationalityLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Track which languages have Wikipedia
        has_wikipedia = []
        for lang, page_data in wiki_pages.items():
            if page_data:
                has_wikipedia.append(lang)

                # Extract metadata
                metadata = page_data.get('metadata', {})

                # Store Wikipedia text with metadata
                db.add_wikipedia_text(
                    entity_id=entity_id,
                    language=lang,
                    title=page_data.get('title'),
                    summary=page_data.get('summary'),
                    content=page_data.get('content'),
                    url=page_data.get('url'),
                    text_length_chars=metadata.get('text_length_chars'),
                    text_length_words=metadata.get('text_length_words'),
                    num_sections=metadata.get('num_sections'),
                    num_references=metadata.get('num_references'),
                    num_backlinks=metadata.get('num_backlinks'),
                )

        # Store Wikipedia availability metadata
        db.add_metadata(entity_id, 'wikipedia_completeness', str(len(has_wikipedia)))
        db.add_metadata(entity_id, 'wikipedia_languages', json.dumps(has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_en', str('en' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_fr', str('fr' in has_wikipedia))
        db.add_metadata(entity_id, 'has_wikipedia_ar', str('ar' in has_wikipedia))
        db.add_metadata(entity_id, 'has_all_wikipedia_languages', str(len(has_wikipedia) == 3))

        collected += 1
        entity_name = person.get('personLabel_en', wikidata_id)
        logger.info(f"Collected person {collected}/{len(valid_people)}: {entity_name}")

    return collected


def collect_events(event_types=None, limit=50, min_year=1800, max_year=2000):
    """Collect event entities with full metadata"""
    logger.info(f"Collecting {limit} events...")

    # Query Wikidata
    events = wikidata.get_events_multilingual(
        event_types=event_types,
        limit=limit,
        min_year=min_year,
        max_year=max_year
    )

    logger.info(f"Got {len(events)} events from Wikidata")

    # Filter for complete labels
    valid_events = [e for e in events if has_complete_labels(e, 'event')]
    logger.info(f"Filtered to {len(valid_events)} with complete labels")

    collected = 0
    for event in valid_events:
        wikidata_id = event.get('event_id')
        if not wikidata_id:
            continue

        # Add base entity
        entity_id = db.add_entity(
            wikidata_id=wikidata_id,
            entity_type='event',
            culture=None,  # Events don't have inherent culture
            country=event.get('locationLabel_en')  # Use location as country
        )

        if not entity_id:
            continue

        # Add labels
        for lang in ['en', 'fr', 'ar']:
            label = event.get(f'eventLabel_{lang}')
            if label:
                db.add_entity_label(entity_id, lang, label)

        # Add metadata
        if event.get('eventType'):
            db.add_metadata(entity_id, 'event_type', event['eventType'])

        if event.get('date'):
            event_year = event['date'][:4]
            db.add_metadata(entity_id, 'event_year', event_year)

        # Fetch Wikipedia language count separately
        num_wiki_langs = wikidata.get_sitelinks_count(wikidata_id)
        db.add_metadata(entity_id, 'num_wikipedia_languages', str(num_wiki_langs))

        # Add facts
        if event.get('date'):
            db.add_fact(entity_id, 'P585', event['date'])

        if event.get('location'):
            fact_id = db.add_fact(entity_id, 'P276', event['location'])
            if fact_id:
                for lang in ['en', 'fr', 'ar']:
                    label = event.get(f'locationLabel_{lang}')
                    if label:
                        db.add_fact_label(fact_id, lang, label)

        # Fetch Wikipedia
        entity_names = {
            'en': event.get('eventLabel_en'),
            'fr': event.get('eventLabel_fr'),
            'ar': event.get('eventLabel_ar')
        }

        wiki_pages = wikipedia.get_page_multilingual(entity_names)

        has_wikipedia = []
        for lang, page_data in wiki_pages.items():
            if page_data:
                has_wikipedia.append(lang)
                metadata = page_data.get('metadata', {})

                db.add_wikipedia_text(
                    entity_id=entity_id,
                    language=lang,
                    title=page_data.get('title'),
                    summary=page_data.get('summary'),
                    content=page_data.get('content'),
                    url=page_data.get('url'),
                    text_length_chars=metadata.get('text_length_chars'),
                    text_length_words=metadata.get('text_length_words'),
                    num_sections=metadata.get('num_sections'),
                    num_references=metadata.get('num_references'),
                    num_backlinks=metadata.get('num_backlinks'),
                )

        # Store Wikipedia availability
        db.add_metadata(entity_id, 'wikipedia_completeness', str(len(has_wikipedia)))
        db.add_metadata(entity_id, 'wikipedia_languages', json.dumps(has_wikipedia))
        db.add_metadata(entity_id, 'has_all_wikipedia_languages', str(len(has_wikipedia) == 3))

        collected += 1
        logger.info(f"Collected event {collected}/{len(valid_events)}: {entity_names['en']}")

    return collected

2025-12-14 23:50:29,976 - src.utils - INFO - Logging initialized
2025-12-14 23:50:29,977 - src.data.database - INFO - Database initialized at data/entities.db


In [2]:
    # Collect Western people
total_western = collect_people(
        nationalities=WESTERN_COUNTRIES,
        limit=5,
        min_year=1850,
        max_year=2000
    )

2025-12-14 23:50:32,659 - __main__ - INFO - Collecting 5 people...
2025-12-14 23:50:32,660 - src.data.wikidata_client - INFO - Querying 4 nationalities (multilingual), limit=5
2025-12-14 23:50:36,347 - src.data.wikidata_client - INFO - Query successful: 5 results
2025-12-14 23:50:36,348 - __main__ - INFO - Got 5 people from Wikidata
2025-12-14 23:50:36,348 - __main__ - INFO - Filtered to 3 with complete labels
2025-12-14 23:50:36,354 - src.data.database - INFO - Added entity: Q26853
2025-12-14 23:50:39,629 - src.data.wikipedia_client - INFO - Fetched Wikipedia page: Camilo Cienfuegos (44985 chars)
2025-12-14 23:50:39,763 - src.data.wikipedia_client - ERROR - Failed to get backlinks for 'Camilo Cienfuegos': 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&list=backlinks&bltitle=Camilo+Cienfuegos&bllimit=1&format=json
2025-12-14 23:50:41,042 - src.data.wikipedia_client - INFO - Fetched Wikipedia page: Camilo Cienfuegos (5275 chars)
2025-12-14 23:50:41,

In [4]:
total_western

3

In [9]:
stats = db.get_statistics()

In [10]:
stats

{'total_entities': 7,
 'by_type': {'person': 7},
 'by_culture': {'Western': 7},
 'labels_by_language': {'ar': 7, 'en': 7, 'fr': 7},
 'wikipedia_by_language': {'ar': 7, 'en': 5, 'fr': 7},
 'total_questions': 0,
 'total_facts': 21}

In [7]:
# Get all entities
entities = db.get_all_entities()

print("=" * 60)
print("COLLECTED ENTITIES")
print("=" * 60)

for entity in entities:
    entity_id = entity['id']
    wikidata_id = entity['wikidata_id']

    # Get labels
    labels = db.get_entity_labels(entity_id)

    # Get metadata
    metadata = db.get_metadata(entity_id)

    # Get Wikipedia texts
    wiki_texts = db.get_wikipedia_texts(entity_id)

    print(f"\nEntity: {labels.get('en', 'N/A')}")
    print(f"  Wikidata ID: {wikidata_id}")
    print(f"  Culture: {entity['culture']}")
    print(f"  Labels:")
    print(f"    EN: {labels.get('en', 'Missing')}")
    print(f"    FR: {labels.get('fr', 'Missing')}")
    print(f"    AR: {labels.get('ar', 'Missing')}")

    print(f"  Metadata:")
    print(f"    Gender: {metadata.get('gender', 'N/A')}")
    print(f"    Birth year: {metadata.get('birth_year', 'N/A')}")
    print(f"    Wikipedia languages: {metadata.get('num_wikipedia_languages', '0')}")
    print(f"    Has all Wikipedia: {metadata.get('has_all_wikipedia_languages', 'False')}")

    print(f"  Wikipedia:")
    for lang in ['en', 'fr', 'ar']:
        if lang in wiki_texts:
            text = wiki_texts[lang]
            print(f"    {lang.upper()}: {text['title']} ({len(text['content'])} chars)")
        else:
            print(f"    {lang.upper()}: Missing")

    # Get facts
    facts = db.get_entity_facts(entity_id)
    if facts:
        print(f"  Facts: {len(facts)} properties")
        for fact in facts[:3]:  # Show first 3
            labels_dict = fact.get('labels', {})
            print(f"    {fact['property']}: {labels_dict.get('en', 'N/A')}")

print("\n" + "=" * 60)

COLLECTED ENTITIES

Entity: David Villa
  Wikidata ID: Q83006
  Culture: Western
  Labels:
    EN: David Villa
    FR: David Villa
    AR: دافيد فيا
  Metadata:
    Gender: http://www.wikidata.org/entity/Q6581097
    Birth year: 1981
    Wikipedia languages: 81
    Has all Wikipedia: False
  Wikipedia:
    EN: Missing
    FR: David Villa (21685 chars)
    AR: دافيد فيا (41118 chars)
  Facts: 3 properties
    P19: Tiuya / Tuilla
    P27: Spain
    P569: N/A

Entity: Willy Brandt
  Wikidata ID: Q2514
  Culture: Western
  Labels:
    EN: Willy Brandt
    FR: Willy Brandt
    AR: ويلي براندت
  Metadata:
    Gender: http://www.wikidata.org/entity/Q6581097
    Birth year: 1913
    Wikipedia languages: 101
    Has all Wikipedia: False
  Wikipedia:
    EN: Missing
    FR: Willy Brandt (11960 chars)
    AR: ويلي براندت (4024 chars)
  Facts: 3 properties
    P19: Lübeck
    P27: Spain
    P569: N/A

Entity: Carlos Cuéllar
  Wikidata ID: Q125410
  Culture: Western
  Labels:
    EN: Carlos Cuéllar

In [8]:
wikidata_id = "Q83006"  # David Villa
entity = db.get_entity(wikidata_id)
entity_id = entity['id']

# Get Wikipedia texts
wiki_texts = db.get_wikipedia_texts(entity_id)

print("=" * 80)
print(f"Wikipedia Articles for: {wikidata_id}")
print("=" * 80)

for lang, text_data in wiki_texts.items():
    print(f"\n{'='*80}")
    print(f"LANGUAGE: {lang.upper()}")
    print(f"{'='*80}")
    print(f"Title: {text_data['title']}")
    print(f"URL: {text_data['url']}")
    print(f"\nSummary:")
    print(text_data['summary'])
    print(f"\nFull Content ({len(text_data['content'])} chars):")
    print(text_data['content'][:500])  # First 500 chars
    print("...")
    print(text_data['content'][-500:])  # Last 500 chars

Wikipedia Articles for: Q83006

LANGUAGE: AR
Title: دافيد فيا
URL: https://ar.wikipedia.org/wiki/%D8%AF%D8%A7%D9%81%D9%8A%D8%AF_%D9%81%D9%8A%D8%A7

Summary:
دافيد فيا سانتشيز (مواليد 3 ديسمبر 1981) هو لاعب كرة قدم إسباني سابق لعب في مركز المهاجم. يُعد واحداً من أفضل المهاجمين في جيله، ويُعتبر الهدّاف التاريخي لمنتخب إسبانيا. يشغل حالياً منصب نائب رئيس نادي بينيدورم.
لُقّب باسم "إل غواخي" أي "الولد" باللهجة الأسترية، بسبب سمعته في اللعب مع أطفال أكبر منه سناً. تعرّض لإصابة خطيرة في طفولته، لكنه نجح لاحقاً في بدء مسيرته الاحترافية مع سبورتينغ خيخون عام 2001. وبعد موسمين انتقل إلى ريال سرقسطة، حيث خاض أولى مبارياته في الدوري الإسباني وفاز بلقبي كأس ملك إسبانيا وكأس السوبر الإسباني. وفي عام 2005 انضم إلى فالنسيا مقابل 12 مليون يورو، وحقق معهم لقب كأس ملك إسبانيا.
وبعد خمسة مواسم مع فالنسيا أصبح خلالها من أبرز لاعبي الدوري الإسباني، وقع دافيد مع برشلونة عام 2010 مقابل 40 مليون يورو. وفي موسمه الأول فاز بلقب الدوري الإسباني ودوري أبطال أوروبا، وسجل في نهائي 2011. غادر النادي عام 2013 بعد انت

In [ ]:
db.get_wikipedia_texts()

In [ ]:

def main():
    """Main collection pipeline"""
    logger.info("Starting data collection...")


    logger.info(f"Collected {total_western} Western people")

    # Collect Arabic people
    total_arabic = collect_people(
        nationalities=ARABIC_COUNTRIES,
        limit=50,
        min_year=1850,
        max_year=2000
    )
    logger.info(f"Collected {total_arabic} Arabic people")

    # Collect Asian people
    total_asian = collect_people(
        nationalities=ASIAN_COUNTRIES,
        limit=50,
        min_year=1850,
        max_year=2000
    )
    logger.info(f"Collected {total_asian} Asian people")

    # Collect events
    total_events = collect_events(limit=30)
    logger.info(f"Collected {total_events} events")

    # Get statistics
    stats = db.get_statistics()
    logger.info("=" * 60)
    logger.info("COLLECTION COMPLETE")
    logger.info("=" * 60)
    logger.info(f"Total entities: {stats['total_entities']}")
    logger.info(f"By type: {stats['by_type']}")
    logger.info(f"By culture: {stats['by_culture']}")
    logger.info(f"Labels by language: {stats['labels_by_language']}")
    logger.info(f"Wikipedia by language: {stats['wikipedia_by_language']}")
    logger.info(f"Total questions: {stats['total_questions']}")
    logger.info(f"Total facts: {stats['total_facts']}")

    # Check Wikipedia completeness
    complete_entities = db.get_entities_with_all_wikipedia()
    logger.info(f"\nEntities with complete Wikipedia (all 3 languages): {len(complete_entities)}")

    db.close()


if __name__ == "__main__":
    main()